# Strategy 4. BioChatter

# BioChatter Prompt Strategy

### Overview

According to the lead developer of BioChatter. in this video: [[https://www.youtube.com/watch?v=CJIzhx0ilbY&ab_channel=OpenBoxScience]] we can hear the lead developer of BioChatter say:

> You could instruct an LLM like ChatGPT by saying, 'I have this knowledge graph,' then paste the entire schema configuration into the prompt and ask it to generate a query to extract all gene-disease relationships. In BioChatter, however, we take a more structured approach. We do multiple individual steps: first, we select the relevant entities, then the properties, then the relationships between entities, and finally, we generate the query. This process significantly increases accuracy across both closed and open-source models.

Thus BioChatter's approach to generating Cypher queries involves a structured, multi-step process, it breaks down query generation into stages. Each stage isolates specific elements—entities, properties, relationships—before building the final query prompt.

### Prompt Strategy Outline
The official github repo for BioChatter lays down the specific text and approach: 
The code snippets and their line references are sourced from [BioChatter on GitHub](https://github.com/biocypher/biochatter/blob/main/biochatter/prompts.py).

1. **Entity Selection**:  
    The prompt asks the LLM to identify relevant entity types based on the user question. Only the entity names are returned, comma-separated, without additional text.

```{python}
conversation.append_system_message(
    (
        "You have access to a knowledge graph that contains "
        f"these entity types: {', '.join(self.entities)}. Your task is "
        "to select the entity types that are relevant to the user's question "
        "for subsequent use in a query. Only return the entity types, "
        "comma-separated, without any additional text. Do not return "
        "entity names, relationships, or properties."
    )
)
```

- _Code Reference_: Lines 330-352
- _Storage_: Results are stored in `selected_entities`.

 2. **Relationship Selection**:  
After identifying entities, the LLM is prompted to choose relationships between them. The selected relationships are used to specify directional roles (source or target) if available.
```{python}
msg = (
    "You have access to a knowledge graph that contains "
    f"these entities: {', '.join(self.selected_entities)}. "
    "Your task is to select the relationships that are relevant "
    "to the user's question for subsequent use in a query. Only "
    "return the relationships without their sources or targets, "
    "comma-separated, and without any additional text. Here are the "
    "possible relationships and their source and target entities: "
    f"{rels}."
)
```
- _Code Reference_: Lines 356-512

3. **Property Selection:**
A further prompt instructs the LLM to choose properties for the query, stored in `selected_properties`. These properties are presented in a compact JSON format.
```{python}
msg = (
    "You have access to a knowledge graph that contains entities and "
    "relationships. They have the following properties. Entities:"
    f"{e_props}, Relationships: {r_props}. "
    "Your task is to select the properties that are relevant to the "
    "user's question for subsequent use in a query. Only return the "
    "entities and relationships with their relevant properties in compact "
    "JSON format, without any additional text. Return the "
    "entities/relationships as top-level dictionary keys, and their "
    "properties as dictionary values. "
    "Do not return properties that are not relevant to the question."
)
```
- _Code Reference_: Lines 523-587

4. **Final Query Generation**
With selected entities, relationships, and properties, the final prompt is crafted to instruct the LLM to generate a query in the specified language (e.g., Cypher). The relationships are expanded to include valid combinations of source, relationship, and target.

```{python}
msg = (
    f"Generate a database query in {query_language} that answers "
    f"the user's question. "
    f"You can use the following entities: {entities}, "
    f"relationships: {list(relationships.keys())}, and "
    f"properties: {properties}. "
)

for relationship, values in relationships.items():
    self._expand_pairs(relationship, values)

if self.rel_directions:
    msg += "Given the following valid combinations of source, relationship, and target: "
    for key, value in self.rel_directions.items():
        for pair in value:
            msg += f"'(:{pair[0]})-(:{key})->(:{pair[1]})', "
    msg += f"generate a {query_language} query using one of these combinations. "

msg += "Only return the query, without any additional text, symbols or characters --- just the query statement."

conversation.append_system_message(msg)

out_msg, token_usage, correction = conversation.query(question)
```

- _Code Reference_: Lines 180-224

In [3]:
import os
from dotenv import load_dotenv
from tqdm import tqdm
import json
import pandas as pd

# Load environment variables from the .env file
load_dotenv()

# Retrieve the variables from the environment
neo4j_uri = os.getenv("NEO4J_URI")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

# Check if any variable is missing
if not all([neo4j_uri, neo4j_username, neo4j_password]):
    raise EnvironmentError("One or more environment variables are missing: NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD")

print(f"Accessing OpenTargets at {neo4j_uri} as user {neo4j_username}")



Accessing OpenTargets at bolt+s://pistoia.neo4j.rbsapp.net:7687 as user neo4j


Wrappers for LLMs and KG

In [4]:
# Langchain wrappers for different models

import re
from langchain_openai import ChatOpenAI
from langchain_mistralai import ChatMistralAI
from langchain_anthropic import ChatAnthropic

chat_models = {}

def new_ChatModel(model):
    if re.search(r"^gpt", model):
        return ChatOpenAI(model = model)
    elif re.search(r"^o1", model):
        return ChatOpenAI(model = model, temperature = 1)
    elif re.search(r"^claude", model):
        return ChatAnthropic(model = model)
    elif re.search(r"mistral", model):
        return ChatMistralAI(model = model)
    else:
        raise ValueError(f"Unsupported model: {model}")

def ChatModel(model):
    if model in chat_models:
        return chat_models[model]
    else:
        chat_models[model] = new_ChatModel(model)
        return chat_models[model]

# models:
#   claude-3-5-sonnet-20240620
#   gpt-4o
#   o1-preview-2024-09-12
#   open-mistral-7b

# llm = ChatModel("gpt-4o")


In [5]:
# Cypher data extraction

def extract_cypher(message):
    text = message
    try:
        pattern = r"```cypher(.*?)```"
        matches = re.findall(pattern, text, re.DOTALL)
        if len(matches) > 0:
            return [match.strip() for match in matches]
        else:
            pattern = r"```(.*?)```"
            matches = re.findall(pattern, text, re.DOTALL)
            return [match.strip() for match in matches]
            
    except Exception:
        raise ValueError(f"Failed to parse: {message}")

from py2neo import Graph

graph = Graph(
    neo4j_uri,
    auth=(neo4j_username, neo4j_password)
)



In [6]:
# Some queries take a very long time to run. This will deal with timeout

import threading

class TimeoutThread(threading.Thread):
    def __init__(self, func, *args, **kwargs):
        threading.Thread.__init__(self)
        self.func = func
        self.args = args
        self.kwargs = kwargs
        self.result = None
        self.error = None

    def run(self):
        try:
            self.result = self.func(*self.args, **self.kwargs)
        except Exception as e:
            self.error = e

def run_with_timeout(func, timeout, *args, **kwargs):
    thread = TimeoutThread(func, *args, **kwargs)
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        raise TimeoutError("Function execution timed out")
    elif thread.error:
        raise thread.error
    return thread.result

In [7]:
# convenience functions for data retrieval from graph

import time

def query_cypher_graph(graph, query):
    return graph.query(query)

def query_graph(llm_output):
    try:
        cypher_query = extract_cypher(llm_output)
    except Exception as e:
        return [{
            "query":None,
            "success":False,
            "exception":str(e)
        }]

    cypher_results = []
    for query in cypher_query:
        try:
            start = time.time()
            result = run_with_timeout(query_cypher_graph, 20, graph, query)
            duration = time.time() - start
            cypher_results.append({
                "query":query,
                "success":True,
                "result": list(result),
                "time": duration
                })
        except Exception as e:
            cypher_results.append({
                "query":query,
                "success":False,
                "exception":str(e)
            })
    return cypher_results


def process_results(todo, llm_answers, cypher_results):
    results = []
    for t,llm,res in zip(todo, llm_answers, cypher_results):
        out = {
            "model" : t[0],
            "question" : t[1],
            "llm_answer" : llm,
            "cypher_output": res,
            "n_cypher_queries" : len(res)
        }
        if len(res) > 0:
            out.update({
                "query": res[0].get('query',''),
                "success": res[0]['success']
            })
            if res[0]['success']:
                out.update({
                    "results": res[0]['result'],
                    "time": res[0]['time'],
                    "count" : len(res[0]['result'])
                })
            else:
                out.update({
                    "error": res[0]['exception']
                })
        results.append(out)
    return results



## Questions

In [8]:
questions = {
    "tdp-als": "What (or how strong, or is there any) is the evidence between TDP-43 and amyotrophic lateral sclerosis (ALS)?",
    "tdp-cancer": "What is the evidence linking TDP-43 to cancer in animal models?",
    "braf-melanoma": "What (or is there) is the clinical evidence linking BRAF to Melanoma?",
}

In [9]:
# Graph schema

from langchain_community.graphs import Neo4jGraph

normal_path = "normal_schema.json"

if os.path.exists(normal_path):
    with open(normal_path, "r") as f:
        # normal_schema = f.read()
        normal_schema = json.load(f)
else:
    normal_graph = Neo4jGraph(
        url=neo4j_uri,
        username=neo4j_username,
        password=neo4j_password,
    )
    normal_graph.refresh_schema()
    normal_schema = normal_graph.schema
    # open(normal_path, "w").write(normal_schema)
    with open(normal_path, "w") as f:
        json.dump(normal_schema, f)

In [10]:
import re

def extract_available_entities(schema_string):
    # Use a regex pattern to match lines that define nodes
    # Node definition lines typically start with "Entity", "ThingWithTaxon", etc.
    pattern = r"([\w\.]+) \{.*?\}"  # Match node label before the first opening curly brace
    matches = re.findall(pattern, schema_string)
    return list(set(matches))  # Remove duplicates if any

# Example usage
available_entities = extract_available_entities(normal_schema)
# for item in available_entities:
#     print(item)
# print(normal_schema)
# print("Available Entities:", available_entities)


In [11]:
def extract_relationships(schema_string):
    # Split the string into lines
    lines = schema_string.splitlines()
    # Find the index of the line containing "The relationships:"
    rl_index = next(i for i, line in enumerate(lines) if "The relationships:" in line)
    # Return all lines after "The relationships:"
    return lines[1:rl_index - 2], lines[rl_index + 1:]

# Example usage
node_properties, relationship_definitions = extract_relationships(normal_schema)
node_properties = "\n".join(node_properties)
relationship_definitions = "\n".join(relationship_definitions)
# print(node_properties)  # Print the node_properties line by line
# print(relationship_definitions)  # Print the relationships line by line


In [12]:
# print(normal_schema)

# Option 4

In [13]:
from langchain.schema import HumanMessage, SystemMessage

models = ["gpt-4o", "claude-3-5-sonnet-20240620", "open-mistral-7b", "o1-preview-2024-09-12"]
# models = ["gpt-4o", "claude-3-5-sonnet-20240620", "o1-preview-2024-09-12"]
#models = ["claude-3-5-sonnet-20240620"]
niter = 10
# niter = 2

todo = [(m, q, i) for q in questions.items() for m in models for i in range(niter)]

In [14]:
import time

def send_llm(llm_model, llm, system_prompt, user_prompt):
    if llm_model == 'o1-preview-2024-09-12':
        messages = f"{system_prompt}\n------------------------------------------\nUser question:\n{user_prompt}\n"
        result = llm.invoke(messages)
    else:
        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt)
        ]
        result = llm.invoke(messages)
    if llm_model == 'open-mistral-7b':
        time.sleep(2)
    return result.content.strip()

In [15]:
query_prompt_template = """
Generate a database query in Cypher that answers the user's question.

You can use the following entities: {selected_entities},
relationships: {selected_relationships}
properties: {selected_properties}.

Generate a Cypher query using one of these relationships.
Only return the query encapsulated in triple backticks with cypher indicating it is a cypher query,
without any additional text, symbols or characters --- just the query statement.

IMPORTNAT:
- Always escape labels containing dots and other not allowed symbols with backticks!
- Make queries case-insensitive.
"""

In [16]:



def run_llm_4(llm_model, question):
    try:
        llm = ChatModel(model=llm_model)

        # Step 1: Entity Selection
        entity_prompt = (
            f"You have access to a knowledge graph that contains "
            f"these entity types: {', '.join(available_entities)}. "
            "Your task is to select the entity types that are relevant to the user's question "
            "for subsequent use in a query. Only return the entity types, comma-separated, "
            "without any additional text."
        )
        selected_entities_response = send_llm(llm_model, llm, entity_prompt, question)
        selected_entities = selected_entities_response.split(",")  # Parse response
        print(f"found the following entities: {selected_entities}")

        # Step 2: Relationship Selection
        relationship_prompt = (
            f"You have access to a knowledge graph that contains "
            f"these entities: {', '.join(selected_entities)}. "
            "Your task is to select the relationships that are relevant "
            "to the user's question for subsequent use in a query. Only "
            "return the relationships that are relevant for the question, "
            "comma-separated, and without any additional text. Here are the "
            "possible relationships and their source and target entities: "
            f"{relationship_definitions}."
        )
        selected_relationships_response = send_llm(llm_model, llm, relationship_prompt, question)
        selected_relationships = selected_relationships_response.split(",")  # Parse response
        print(f"found the following relationships: {selected_relationships} for the entities selected {selected_entities}")

        # Step 3: Property Selection
        property_prompt = (
            "You have access to a knowledge graph that contains entities and "
            "relationships. They have the following properties. Entities:"
            f"{node_properties}. "
            "Your task is to select the properties that are relevant to the "
            "user's question for subsequent use in a query. Only return the "
            "entities with their relevant properties "
            "without any additional text."
        )
        selected_properties_response = send_llm(llm_model, llm, property_prompt, question)
        selected_properties = selected_properties_response
        print(f"found the following properties: {selected_properties_response}")

        # Step 4: Final Query Generation
        query_prompt = query_prompt_template.format(selected_entities=selected_entities, selected_relationships = selected_relationships, selected_properties = selected_properties)
        final_query_response = send_llm(llm_model, llm, query_prompt, question)
        print(f"Got the following query: {final_query_response}")


        return final_query_response  # Return the final Cypher query

    except Exception as e:
        print(e)
        return None


In [21]:
llm_answers = []
for llm_model, question, iter in tqdm(todo, desc="Prompting LLM"):
    file_p = f'../../data/{llm_model}_{iter}_{question[0]}.txt'
    if os.path.exists(file_p):
        with open(file_p, "r") as f:
            llm_answers.append(f.read())
    else:
        print(f"Model: {llm_model}, Prompt: {question[1]}, iter {iter}")
        answer = run_llm_4(llm_model, question[1])
        if answer:
            with open(file_p, "w") as f:
                f.write(answer)
        llm_answers.append(answer)

Prompting LLM: 100%|██████████| 120/120 [00:00<00:00, 642.34it/s]


In [22]:
cypher_results = []
for answer in tqdm(llm_answers, desc="Querying graph"):
    cypher_results.append(query_graph(answer))

Querying graph: 100%|██████████| 120/120 [00:52<00:00,  2.30it/s]


In [23]:
results = process_results(todo, llm_answers, cypher_results)
results_df = pd.DataFrame(results)
results_df

,model,question,llm_answer,cypher_output,n_cypher_queries,query,success,results,time,count,error
0,gpt-4o,"(tdp-als, What (or how strong, or is there any...",```cypher\nMATCH (g:HumanGene)-[:IS_PART_OF]->...,[{'query': 'MATCH (g:HumanGene)-[:IS_PART_OF]-...,1,MATCH (g:HumanGene)-[:IS_PART_OF]->(assoc:Gene...,True,[],0.208170,0.0,NaN
1,gpt-4o,"(tdp-als, What (or how strong, or is there any...",```cypher\nMATCH (g:Gene)-[:IS_PART_OF]->(a:Ge...,[{'query': 'MATCH (g:Gene)-[:IS_PART_OF]->(a:G...,1,MATCH (g:Gene)-[:IS_PART_OF]->(a:GeneToDisease...,True,[],0.168253,0.0,NaN
2,gpt-4o,"(tdp-als, What (or how strong, or is there any...",```cypher\nMATCH (g:Gene)-[:IS_PART_OF]->(asso...,[{'query': 'MATCH (g:Gene)-[:IS_PART_OF]->(ass...,1,MATCH (g:Gene)-[:IS_PART_OF]->(assoc:GeneToDis...,True,[],0.172263,0.0,NaN
3,gpt-4o,"(tdp-als, What (or how strong, or is there any...",```cypher\nMATCH (g:Gene)-[:IS_PART_OF]->(a:`G...,[{'query': 'MATCH (g:Gene)-[:IS_PART_OF]->(a:`...,1,MATCH (g:Gene)-[:IS_PART_OF]->(a:`GeneToDiseas...,True,[],0.170810,0.0,NaN
4,gpt-4o,"(tdp-als, What (or how strong, or is there any...",```cypher\nMATCH (g:Gene)-[:IS_PART_OF]->(a:`G...,[{'query': 'MATCH (g:Gene)-[:IS_PART_OF]->(a:`...,1,MATCH (g:Gene)-[:IS_PART_OF]->(a:`GeneToDiseas...,True,[],0.214822,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
115,o1-preview-2024-09-12,"(braf-melanoma, What (or is there) is the clin...",```cypher\nMATCH (g:Gene)-[:IS_PART_OF]->(asso...,[{'query': 'MATCH (g:Gene)-[:IS_PART_OF]->(ass...,1,"MATCH (g:Gene)-[:IS_PART_OF]->(assoc),\n ...",True,"[({'score': 0.2, 'licence': 'CC BY-SA 3.0', 'p...",0.629994,4205.0,NaN
116,o1-preview-2024-09-12,"(braf-melanoma, What (or is there) is the clin...",```cypher\nMATCH (g:HumanGene)-[:IS_PART_OF]->...,[{'query': 'MATCH (g:HumanGene)-[:IS_PART_OF]-...,1,MATCH (g:HumanGene)-[:IS_PART_OF]->(assoc)<-[:...,True,"[(None, 0.7, eva_somatic), (None, 0.7, eva_som...",0.335731,4059.0,NaN
117,o1-preview-2024-09-12,"(braf-melanoma, What (or is there) is the clin...",```cypher\nMATCH (g:Gene)-[:IS_PART_OF]->(gda:...,[{'query': 'MATCH (g:Gene)-[:IS_PART_OF]->(gda...,1,MATCH (g:Gene)-[:IS_PART_OF]->(gda:`Literature...,True,"[([26911405], 0.02), ([29175850], 0.13), ([226...",0.291903,4011.0,NaN
118,o1-preview-2024-09-12,"(braf-melanoma, What (or is there) is the clin...",```cypher\nMATCH (g:Gene)-[:IS_PART_OF]->(a:`K...,[{'query': 'MATCH (g:Gene)-[:IS_PART_OF]->(a:`...,1,MATCH (g:Gene)-[:IS_PART_OF]->(a:`KnownDrug.Ge...,True,"[(chembl, 0.2, 5dcc2d37a4d7131e14636de210aecfa...",0.209071,146.0,NaN


In [24]:
# Function to clean illegal characters
def clean_illegal_characters(value):
    if isinstance(value, str):
        # Remove control characters
        value = re.sub(r"[\x00-\x09\x0B-\x1F\x7F-\x9F]", "", value)
        # Replace multiline text with a single-line equivalent
        # value = value.replace("\n", " ").replace("\r", " ")
    return value

# Apply cleaning to the entire DataFrame
results_df = results_df.apply(lambda x: x.map(clean_illegal_characters))
# # Save to Excel
# try:
#     results_df.to_excel("04-evaluations.xlsx", index=False)
#     print("File saved successfully!")
# except Exception as e:
#     print(f"Error saving file: {e}")


In [25]:
results_df.to_excel("04c-evaluations.xlsx", index=False)


In [26]:
with open("04c-evaluations.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)
results_df

,model,question,llm_answer,cypher_output,n_cypher_queries,query,success,results,time,count,error
0,gpt-4o,"(tdp-als, What (or how strong, or is there any...",```cypher\nMATCH (g:HumanGene)-[:IS_PART_OF]->...,[{'query': 'MATCH (g:HumanGene)-[:IS_PART_OF]-...,1,MATCH (g:HumanGene)-[:IS_PART_OF]->(assoc:Gene...,True,[],0.208170,0.0,NaN
1,gpt-4o,"(tdp-als, What (or how strong, or is there any...",```cypher\nMATCH (g:Gene)-[:IS_PART_OF]->(a:Ge...,[{'query': 'MATCH (g:Gene)-[:IS_PART_OF]->(a:G...,1,MATCH (g:Gene)-[:IS_PART_OF]->(a:GeneToDisease...,True,[],0.168253,0.0,NaN
2,gpt-4o,"(tdp-als, What (or how strong, or is there any...",```cypher\nMATCH (g:Gene)-[:IS_PART_OF]->(asso...,[{'query': 'MATCH (g:Gene)-[:IS_PART_OF]->(ass...,1,MATCH (g:Gene)-[:IS_PART_OF]->(assoc:GeneToDis...,True,[],0.172263,0.0,NaN
3,gpt-4o,"(tdp-als, What (or how strong, or is there any...",```cypher\nMATCH (g:Gene)-[:IS_PART_OF]->(a:`G...,[{'query': 'MATCH (g:Gene)-[:IS_PART_OF]->(a:`...,1,MATCH (g:Gene)-[:IS_PART_OF]->(a:`GeneToDiseas...,True,[],0.170810,0.0,NaN
4,gpt-4o,"(tdp-als, What (or how strong, or is there any...",```cypher\nMATCH (g:Gene)-[:IS_PART_OF]->(a:`G...,[{'query': 'MATCH (g:Gene)-[:IS_PART_OF]->(a:`...,1,MATCH (g:Gene)-[:IS_PART_OF]->(a:`GeneToDiseas...,True,[],0.214822,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
115,o1-preview-2024-09-12,"(braf-melanoma, What (or is there) is the clin...",```cypher\nMATCH (g:Gene)-[:IS_PART_OF]->(asso...,[{'query': 'MATCH (g:Gene)-[:IS_PART_OF]->(ass...,1,"MATCH (g:Gene)-[:IS_PART_OF]->(assoc),\n ...",True,"[({'score': 0.2, 'licence': 'CC BY-SA 3.0', 'p...",0.629994,4205.0,NaN
116,o1-preview-2024-09-12,"(braf-melanoma, What (or is there) is the clin...",```cypher\nMATCH (g:HumanGene)-[:IS_PART_OF]->...,[{'query': 'MATCH (g:HumanGene)-[:IS_PART_OF]-...,1,MATCH (g:HumanGene)-[:IS_PART_OF]->(assoc)<-[:...,True,"[(None, 0.7, eva_somatic), (None, 0.7, eva_som...",0.335731,4059.0,NaN
117,o1-preview-2024-09-12,"(braf-melanoma, What (or is there) is the clin...",```cypher\nMATCH (g:Gene)-[:IS_PART_OF]->(gda:...,[{'query': 'MATCH (g:Gene)-[:IS_PART_OF]->(gda...,1,MATCH (g:Gene)-[:IS_PART_OF]->(gda:`Literature...,True,"[([26911405], 0.02), ([29175850], 0.13), ([226...",0.291903,4011.0,NaN
118,o1-preview-2024-09-12,"(braf-melanoma, What (or is there) is the clin...",```cypher\nMATCH (g:Gene)-[:IS_PART_OF]->(a:`K...,[{'query': 'MATCH (g:Gene)-[:IS_PART_OF]->(a:`...,1,MATCH (g:Gene)-[:IS_PART_OF]->(a:`KnownDrug.Ge...,True,"[(chembl, 0.2, 5dcc2d37a4d7131e14636de210aecfa...",0.209071,146.0,NaN
